# Using sensors in Arritmic3D

In this notebook we will show how we can use sensors to record the temporal evolution of the simulation in specific points of the tissue.

We start by importing the arritmic3d module and other required packages.

In [ ]:
%pip install arritmic3d
%pip install pandas

In [ ]:
import os
import time
import shutil
import pyvista as pv
import matplotlib.pyplot as plt
import pandas as pd
import glob
try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False
import arritmic3d as a3d

# Flag to download the simulated cases.
# Set to True if you want the results to
# be downloaded, for inspection with paraview
download_cases = True

# Disable download if not in colab
if not in_colab:
    download_cases = False


We define basic plotting functions that we will use to visualize the results.

In [ ]:
def plot_grid(grid,field="AP",plt_show=False,title = "") :
    rng = ranges.get(field,None)
    grid = grid.threshold(0.5, scalars="restitution_model", all_scalars=True)
    plotter = pv.Plotter(off_screen=True)
    plotter.add_mesh(grid, scalars=field, cmap="coolwarm", show_edges=True,rng=rng)
    plotter.view_xy()
    img = plotter.show(screenshot=True)
    plt.imshow(img)
    plt.axis('off')
    if title != "":
        plt.title(title)
    if plt_show:
        plt.show()
    return img

def plot_vtk(file_path, field="AP",plt_show=False,title = ""):
    grid = pv.read(file_path)
    plot_grid(grid,field=field,plt_show=plt_show,title=title)

ranges = {
    "AP": [-80, 40],
    "APD": [100,400],
    "CV": [0.1,0.8],
    "State": [0, 2],
    "sensor": [0, 1]
}

def download_case(case_path):
    print(" Packaging results...")
    if os.path.exists(case_path) and len(os.listdir(case_path)) > 0:
        zip_name = "/content/case.zip"
        !zip -q -FSr {zip_name} {case_path}
        print(" Downloading results...")
        files.download(zip_name)
        print(f" DOWNLOAD STARTED: {zip_name}")
    else:
        print(" Error: Simulation produced no files.")

## Build a slab with sensors

We will configure the same case as the beginning of the second tutorial, but we will add a new field named `sensor`.

By default we will initialize the `sensor` field to `0` using the `--field` option.
Then, we will add a couple of small regions, spanning just one node, where the `sensor` field is set to `1`. This tells the simulation to record data at these locations. We will place one in the center and another near the top of the slab.

In [ ]:
output_dir="test_sensors"

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(os.path.join(output_dir, "input_data"), exist_ok=True)
slab_path = os.path.join(output_dir, "input_data", "slab.vtk")

# The node spacing is 0.1, so a radius of 0.05 ensures the region covers about one node.
slab_args = [
    slab_path,
    "--nnodes", "70", "70", "2",
    "--spacing", "0.1", "0.1", "0.1",
    "--region-by-side", "south", "1",
    "--field", "restitution_model", "2",
    "--region", '{"shape" : "square", "cx" : 3.5, "cy" : 3.5, "r1" : 2.0, "r2" : 2.0, "restitution_model" : 5}',
    "--field", "sensor", "0",
    "--region", '{"shape" : "square", "cx" : 3.5, "cy" : 3.5, "r1" : 0.05, "r2" : 0.05, "sensor" : 1}',
    "--region", '{"shape" : "square", "cx" : 3.5, "cy" : 6.5, "r1" : 0.05, "r2" : 0.05, "sensor" : 1}'
]

a3d.build_slab(args_list=slab_args)

# Plot the sensor field to see our points
plot_vtk(slab_path, field="sensor", title="Sensors")

## Run the simulation

Now we set the simulation parameters and run it. The simulation will generate `csv` files for each sensor node containing the temporal evolution of the variables at that specific position, as described in the [arritmic3d documentation](https://commlabuv.github.io/arritmic3d/).

In [ ]:
print("\n--- Configuring simulation ---")
config = {
    "VTK_INPUT_FILE": slab_path,
    "APD_MODEL": "TenTuscher",
    "CV_MODEL": "TenTuscher",
    "SIMULATION_DURATION": 5000,
    "VTK_OUTPUT_PERIOD": 10,
    "PROTOCOL": [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [13],
            "BCL": [400]
        }
    ]
}

a3d.arritmic3d(output_dir,config)

if download_cases:
    download_case(output_dir)

## Viewing the sensor results

Inside the `test_sensors/sensors` directory, there will be CSV files corresponding to the points where the `sensor` field was set to 1. 
We can load one of them to plot the Action Potential Duration (APD) and Conduction Velocity (CV) over time, filtering only the rows that correspond to activations to see how they stabilize.

Let's first take a look at the data.

In [ ]:
sensor_files = glob.glob(os.path.join(output_dir, "sensors", "*.csv"))

if sensor_files:
    # Read the first sensor file
    df = pd.read_csv(sensor_files[0])

    print(f"Loaded file: {os.path.basename(sensor_files[0])}")
    print("Dataframe shape:", df.shape)
    display(df.head())
else:
    print("No sensor files found.")
    df = None

In [ ]:
if df is not None:
    # Filter rows to keep only one per activation.
    # We detect activations when 'State' changes to 1 (depolarization)
    activations = df[(df['event_type'] == 1) & (df['event_type'].shift(1) != 1)]

    # Plot APD over time
    plt.figure(figsize=(10, 4))
    plt.plot(activations['Time'], activations['APD'], marker='o', linestyle='-')
    plt.title(f"APD over time ({os.path.basename(sensor_files[0])})")
    plt.xlabel("Time (ms)")
    plt.ylabel("APD (ms)")
    plt.grid(True)
    plt.show()

    # Plot CV over time
    plt.figure(figsize=(10, 4))
    plt.plot(activations['Time'], activations['conduction_velocity'], marker='o', linestyle='-', color='orange')
    plt.title(f"CV over time ({os.path.basename(sensor_files[0])})")
    plt.xlabel("Time (ms)")
    plt.ylabel("CV (cm/s)")
    plt.grid(True)
    plt.show()